# STAC AI Workshop · NYC · 21 October 2026

## 01 · What can you trust when you change architecture?

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Compare fixed RAG, fixed RAG plus calculator, and agent-selected tools
2. Distinguish correct arithmetic from justified inputs
3. Follow an annex dependency and test a programmed branch
4. Identify what a calculation does not establish

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

In [1]:
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

Offline workbook ready: no API key, Node.js, GPU or model download.


## Class Exercise · same documents, different control flow
This is the **same scripted walkthrough as Web Section 1**. It does not measure live model ability. The agent follow-up is deliberately supplied.

Harbor Metals: debt $75m, total cash $15m, EBITDA $20m; ceiling 3.5×.

| Architecture | Who chooses the steps? | What still needs checking? |
|---|---|---|
| Fixed RAG | Application retrieves, model drafts | Right source and supported interpretation |
| Fixed RAG + calculator | Application retrieves and calculates | Formula and input justification |
| Agent-selected tools | Illustrated model requests retrieval/calculation | Necessary steps completed and inputs justified |

In [2]:
def compare(case,branch=False):
    rows=architecture(case,branch)
    show_table([{"Approach":r["label"],"Calculated ratio":f'{r["leverage"]:.2f}×' if r["leverage"] is not None else "No verified calculation","Supported?":r["supported"],"Status":r["status"]} for r in rows])
    return rows
simple=compare("simple")
assert simple[1]["leverage"]==simple[2]["leverage"]==3.0
assert not simple[0]["supported"]

Approach,Calculated ratio,Supported?,Status
Fixed RAG,No verified calculation,False,HOLD CALCULATION CLAIM
Fixed RAG + calculator,3.00×,True,CALCULATION SUPPORTED
Agent-selected tools,3.00×,True,CALCULATION SUPPORTED


## 2 · Follow the annex
The covenant excludes restricted cash. Annex A says $5m is restricted. Initial fixed retrieval does not fetch the annex. The scripted agent does.

**Predict:** can 3.00× be arithmetically correct but unsupported as covenant leverage?

In [3]:
annex=compare("annex")
assert annex[1]["leverage"]==3.0 and not annex[1]["supported"]
assert annex[2]["leverage"]==3.25 and annex[2]["supported"]
show_table([{"Dependency":"Filing → total cash","Evidence":"$15m"},{"Dependency":"Covenant → Annex A","Evidence":"Restricted cash $5m"},{"Dependency":"Eligible cash → calculator","Evidence":"$15m − $5m = $10m"},{"Dependency":"Calculator → current claim","Evidence":"($75m − $10m) / $20m = 3.25×"}])

Approach,Calculated ratio,Supported?,Status
Fixed RAG,No verified calculation,False,HOLD CALCULATION CLAIM
Fixed RAG + calculator,3.00×,False,HOLD CALCULATION CLAIM
Agent-selected tools,3.25×,True,CALCULATION SUPPORTED


Dependency,Evidence
Filing → total cash,$15m
Covenant → Annex A,Restricted cash $5m
Eligible cash → calculator,$15m − $5m = $10m
Calculator → current claim,($75m − $10m) / $20m = 3.25×


## 3 · Give the fixed workflow a programmed follow-annex branch
Same evidence standard, different control flow. Does the fixed workflow now reach the supported result?

In [4]:
branched=compare("annex",branch=True)
assert branched[1]["supported"] and branched[1]["leverage"]==3.25
assert branched[1]["leverage"]==branched[2]["leverage"]

Approach,Calculated ratio,Supported?,Status
Fixed RAG,No verified calculation,False,HOLD CALCULATION CLAIM
Fixed RAG + calculator,3.25×,True,CALCULATION SUPPORTED
Agent-selected tools,3.25×,True,CALCULATION SUPPORTED


## 4 · Annex unavailable
Neither a calculator nor a flexible investigation can manufacture missing evidence.

In [5]:
missing=compare("missing",branch=True)
assert all(not r["supported"] for r in missing)
print("3.00× is provisional total-cash arithmetic; it is not supported covenant leverage.")
print("Even the supported 3.25× does not establish comfortable headroom.")
print("Downside EBITDA $17m gives",round((75-10)/17,2),"×; scenario assessment and approval still required.")

Approach,Calculated ratio,Supported?,Status
Fixed RAG,No verified calculation,False,HOLD CALCULATION CLAIM
Fixed RAG + calculator,3.00×,False,HOLD CALCULATION CLAIM
Agent-selected tools,3.00×,False,HOLD CALCULATION CLAIM


3.00× is provisional total-cash arithmetic; it is not supported covenant leverage.
Even the supported 3.25× does not establish comfortable headroom.
Downside EBITDA $17m gives 3.82 ×; scenario assessment and approval still required.


## Check against the browser fixtures
These expected outputs were exported from the browser engine. The Python implementation recomputes the results; it does not just display the export.

In [6]:
for ref in load_fixture("architecture-reference.json"):
    for got,want in zip(architecture(ref["caseId"],ref["branch"]),ref["cards"]):
        assert all(got[k]==want[k] for k in got), (ref["caseId"],got)
print("PASS: all 6 case/branch combinations match the web exercise.")

PASS: all 6 case/branch combinations match the web exercise.


## Learning check
1. Does adding a calculator establish that the inputs satisfy the contract? **No.**
2. Does the agent label establish it found the right evidence? **No.**
3. What distinguishes the supported 3.25×? **The source, definition, input binding and arithmetic checks pass.**

## Main takeaway
Retrieval supplies documents; a calculator supplies arithmetic; an agent chooses steps. The evidence graph exposes what was checked and what remains unresolved. This scripted comparison does not establish live agent superiority.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.